# Point-JEPA-style event-camera SSL

Mirror of Point-JEPA's WACV-2025 recipe (Saito & Poovvancheri, arXiv 2404.16432) adapted for event clouds. **No VICReg, no extra regularizer.** What makes the unregularized recipe work where our per-event xattn-pool variant failed:

| Piece | Point-JEPA | Why it's load-bearing |
|---|---|---|
| **Patch tokenization** | mini-PointNet over K nearest points to FPS centroid → 1 patch token | Each token aggregates ~32 events → ~32 bits per token, content-rich. The 1-bit-per-token failure mode that killed the xattn run doesn't apply here. |
| **EMA target** | Momentum ramp `τ ∈ [0.9995, 0.99999]` over training | Standard JEPA stop-grad signal |
| **Target LayerNorm per token, BEFORE loss** | `target = LN(target_encoder(x), dim=-1)` then Smooth-L1 | i-JEPA's actual anti-collapse mechanism. Per-token unit-variance prevents the encoder from drifting to a constant direction. |
| **Predictor with target-position queries** | Mask token + `pos(target_center)` queries cross-attending over encoded context tokens | Standard i-JEPA predictor |
| **Multi-block masking** | 4 target blocks (contiguous in greedy-NN order), context = remaining real tokens | i-JEPA contract |

Architecture in our repo:
- `OmniBirdPatchDataset` provides 256 patch centroids per sample with `ctx_patch_idx` (~64 context) and `tgt_patch_idx` (64 target).
- `PatchOmniBirdEncoder` = Patchifier (mini-PointNet) + dense Transformer over patch tokens.
- `PerceiverPredictor` = standard i-JEPA-style cross-attention predictor.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig,
    PatchOmniBirdEncoder, PerceiverPredictor,
    OmniBirdPatchDataset,
    TargetCenter, ema_update, make_momentum_schedule,
    save_atomic, ensure_dir, short_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3
cfg.signal_dim      = 2                # one-hot ON/OFF
cfg.side            = 64
cfg.n_events_total  = 0
cfg.n_events_max    = 8192             # P_max = 256
cfg.pos_embed       = "gaussian"       # default trainable GFF (Point-JEPA uses trainable sin pos too)
cfg.nerf_freqs      = 5

# Patch layout
cfg.patch_mode         = True
cfg.patch_size         = 32
cfg.patches_per_block  = 16
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block      # 64
cfg.ctx_max_patches    = 128
cfg.patch_curve        = "hilbert"

# Encoder / predictor sizes
cfg.d_model          = 256
cfg.n_layers_enc     = 6
cfg.n_heads          = 8
cfg.dim_head         = 32
cfg.d_pred           = 192
cfg.n_layers_pred    = 4
cfg.n_heads_pred     = 6
cfg.dim_head_pred    = 32
cfg.ffn_mult         = 4
cfg.fourier_dim      = 96
cfg.fourier_scale    = 15.0
cfg.predictor_pos_symmetric = True

# Loss: Smooth-L1 with per-token target LayerNorm (Point-JEPA recipe)
cfg.ema_start        = 0.9995
cfg.ema_end          = 0.99999
cfg.smooth_l1_beta   = 2.0

# Optim
cfg.epochs         = 100
cfg.batch_size     = 64
cfg.lr             = 1e-3
cfg.weight_decay   = 0.05
cfg.warmup_epochs  = 5
cfg.probe_interval = 10
cfg.probe_epochs   = 3
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_pointjepa"
ensure_dir(cfg.ckpt_dir)

print(f"P_max = {cfg.n_events_max // cfg.patch_size}  K = {cfg.patch_size}")
print(f"context = {cfg.ctx_max_patches} patches  target = {cfg.n_tgt} patches")


## 2. CIFAR10-DVS dataset

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root; self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0: ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return out, label


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)

mk = lambda b, tr: OmniBirdPatchDataset(b, cfg, train=tr,
                                          patches_per_block=cfg.patches_per_block,
                                          n_pred_blocks=cfg.n_pred_blocks,
                                          ctx_max=cfg.ctx_max_patches,
                                          patch_size=cfg.patch_size,
                                          curve=cfg.patch_curve)
train_ds, train_eval_ds, test_ds = mk(base_train, True), mk(base_train, False), mk(base_test, False)
train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}")


## 3. Models

In [ ]:
def make_patch_encoder():
    return PatchOmniBirdEncoder(
        d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
        n_heads=cfg.n_heads, dim_head=cfg.dim_head, ffn_mult=cfg.ffn_mult,
        signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
        fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
        patch_size=cfg.patch_size,
    )

context_encoder = make_patch_encoder().to(DEVICE)
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)

predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
    pos_embed=cfg.pos_embed, nerf_freqs=cfg.nerf_freqs,
).to(DEVICE)

# DINO-style target centering: EMA of the per-feature batch mean, subtracted
# from h_tgt_raw BEFORE the per-token LayerNorm. This is what the working
# PBB v2 notebook does and what pointjepa was missing.
#
# Per-token LayerNorm alone allows the "every target points in the same
# direction across batch" minimum — every token has unit variance per dim,
# but they're all colinear. Centering subtracts the running batch mean first
# so a constant-direction target collapses to near-zero (and LN then
# amplifies whatever noise is left into different directions). The trivial
# minimum is no longer satisfiable.
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 4. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
warmup_steps = cfg.warmup_epochs * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

LAST = os.path.join(cfg.ckpt_dir, "pointjepa_last.pt")
BEST = os.path.join(cfg.ckpt_dir, "pointjepa_best.pt")
RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = cfg.ema_start
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 5. Probe

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def gather_ctx(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_kpm   = batch["pool_patch_kpm"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
    B, P, K, Fdim = pool_ev.shape
    Pc = ctx_idx.size(1)
    sub_ev    = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(B, Pc, K, Fdim))
    sub_cen   = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(B, Pc, pool_cen.size(-1)))
    sub_kpm   = torch.gather(pool_kpm,   1, ctx_idx)
    sub_evkpm = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(B, Pc, K))
    return sub_ev, sub_cen, sub_kpm, sub_evkpm


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    def _z(batch):
        sub_ev, sub_cen, sub_kpm, sub_evkpm = gather_ctx(batch)
        with torch.no_grad():
            g = enc(sub_ev, sub_cen, patch_kpm=sub_kpm, kpm_per_event=sub_evkpm)
        mask = (~sub_kpm).unsqueeze(-1).float()
        return (g * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_z(b)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_z(b)).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 6. Training loop — Point-JEPA recipe

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            pool_ev    = batch["pool_patch_events"].to(DEVICE)         # (B, P, K, F)
            pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
            pool_kpm   = batch["pool_patch_kpm"].to(DEVICE)
            pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
            tgt_idx    = batch["tgt_patch_idx"].to(DEVICE)
            tgt_cen    = batch["tgt_patch_centroids"].to(DEVICE)

            # Target: encode ALL patches → gather target patches → CENTER → LN.
            # Two-stage normalization (as in the working PBB v2 notebook):
            # (1) subtract an EMA of the per-feature batch mean (DINO centering),
            # (2) per-token LayerNorm. Centering removes the constant-direction
            # collapse minimum that per-token LN alone cannot prevent.
            with torch.no_grad():
                g_tgt_all = target_encoder(pool_ev, pool_cen,
                                            patch_kpm=pool_kpm, kpm_per_event=pool_evkpm)
                                                                         # (B, P, D)
                Bg, Pg, Dg = g_tgt_all.shape
                h_tgt_raw = torch.gather(g_tgt_all, 1,
                                          tgt_idx.unsqueeze(-1).expand(Bg, tgt_idx.size(1), Dg))
                target_center.update(h_tgt_raw)                          # update EMA mean
                h_tgt = F.layer_norm(target_center(h_tgt_raw), (Dg,))    # center then LN

            # Context: encode only context patches
            ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
            Bc, Pc     = ctx_idx.shape
            sub_ev     = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(Bc, Pc,
                                                       pool_ev.size(2), pool_ev.size(3)))
            sub_cen    = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(Bc, Pc, pool_cen.size(-1)))
            sub_kpm    = torch.gather(pool_kpm,   1, ctx_idx)
            sub_evkpm  = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(Bc, Pc, pool_evkpm.size(-1)))
            g_ctx      = context_encoder(sub_ev, sub_cen,
                                          patch_kpm=sub_kpm, kpm_per_event=sub_evkpm)
                                                                          # (B, Pc, D)

            # Predictor: mask-token + target-position queries cross-attend over g_ctx
            h_pred = predictor(g_ctx, tgt_cen,
                                ctx_coords=sub_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=sub_kpm)               # (B, n_tgt, D)

            loss = F.smooth_l1_loss(h_pred, h_tgt, beta=cfg.smooth_l1_beta)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                # Collapse alarms à la Point-JEPA: pred_std and target_std
                pred_std = h_pred.std(dim=0).mean().item()
                tgt_std  = h_tgt.std(dim=0).mean().item()
                cos = F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item()
                print(f"[ep{epoch:02d} st{global_step:05d}]  loss={loss.item():.4f}  "
                      f"pred_std={pred_std:.3f}  tgt_std={tgt_std:.3f}  cos={cos:.3f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}  m={m:.5f}")
                history["diag_steps"].append(global_step)
                history["diag_log"].append({"pred_std": pred_std, "tgt_std": tgt_std,
                                              "cos": cos, "loss": loss.item()})

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        save_atomic(state, LAST)
        if improved: save_atomic(state, BEST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  loss={avg:.4f}  m={m:.5f}  "
              f"{time.time()-t0:.1f}s" + ("  ★" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_p = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_p:.1f}s)")

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 7. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("smooth_L1"); axes[0].set_title("JEPA loss")
    axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    axes[1].plot(steps, [d["pred_std"] for d in history["diag_log"]], label="pred_std", color="C0")
    axes[1].plot(steps, [d["tgt_std"]  for d in history["diag_log"]], label="tgt_std",  color="C2")
    axes[1].axhline(0.01, color='red', ls='--', alpha=0.4, label="pred_std collapse floor")
    axes[1].axhline(0.10, color='orange', ls='--', alpha=0.4, label="tgt_std collapse floor")
    axes[1].set_xlabel("step"); axes[1].set_title("collapse alarms (Point-JEPA)")
    axes[1].set_yscale("log"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
